# KOLF 数据准备与检查

本 Notebook 负责两件事：查看 KOLF `.h5ad` 的元数据，以及在 backed 模式下按小批次拆分 control（NTC）和 perturbation 细胞。

- 不会把完整表达矩阵一次性读入内存。
- 每个批次单独写成临时 `.h5ad`，最后使用 `concat_on_disk` 合并。
- 如果只想查看数据字段，运行第 1–2 节即可；拆分数据会产生大量磁盘 I/O 和临时文件。

注意：下面默认使用 `KOLF_Metabolic_Enzymes_QC_Filtered.h5ad`。如果你实际要用 Pan Genome 数据，请只修改 `file_path`，并确认磁盘空间足够。

## 1. 导入依赖并设置路径

In [ ]:
from pathlib import Path
import gc

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from anndata.experimental import concat_on_disk

# 原始 KOLF 数据文件；如需使用 Pan Genome 版本，请在这里更改
file_path = Path(
    r"E:\\1Personal\\New\\科研起步\\large_genetic_perturbation_20260909"
    r"\\02_kolf_ipsc\\KOLF_Metabolic_Enzymes_QC_Filtered.h5ad"
)

# 输出目录：control / perturbation 拆分文件和临时分块文件
output_dir = file_path.parent / "PrePR_CT_split"
temp_dir = output_dir / "temp_chunks"
output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(parents=True, exist_ok=True)

# 每批处理的细胞数。数值越小越省内存，但会产生更多临时文件
batch_size = 50

print(f"输入文件：{file_path}")
print(f"输出目录：{output_dir}")


## 2. 以 backed 模式查看数据元信息

In [ ]:
# backed='r' 只把矩阵保留在磁盘上；obs / var 元信息可直接查看
adata = sc.read_h5ad(file_path, backed="r")

print("数据规模（细胞数 × 基因数）：", adata.shape)
print("\\n基因注释字段：", adata.var.columns.tolist())
print("细胞注释字段：", adata.obs.columns.tolist())
display(adata.obs.head())
display(adata.var.head())

# KOLF 文件中用于区分对照和扰动的字段应为 gene_target，NTC 表示对照
if "gene_target" not in adata.obs.columns:
    adata.file.close()
    raise KeyError("adata.obs 中没有 gene_target 字段，请检查当前输入文件是否为预期的 KOLF 数据。")

print("\\n各 gene_target 细胞数（前 20 项）：")
display(adata.obs["gene_target"].astype(str).value_counts().head(20))


## 3. 按 NTC / 非 NTC 分组并分批写出

In [ ]:
# 根据 gene_target 标记对照细胞和扰动细胞
targets = adata.obs["gene_target"].astype(str).to_numpy()
groups = {
    "control": np.flatnonzero(targets == "NTC"),
    "perturbation": np.flatnonzero(targets != "NTC"),
}

for name, indices in groups.items():
    print(f"{name}: {len(indices):,} cells")

# 逐组、逐批从原始 h5ad 读取稀疏矩阵并写入磁盘
for group_name, indices in groups.items():
    group_dir = temp_dir / group_name
    group_dir.mkdir(parents=True, exist_ok=True)
    chunk_files = []

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]

        # 只读取当前批次的表达矩阵，避免一次性载入整个 X
        X_batch = csr_matrix(adata.X[batch_idx, :])
        batch = ad.AnnData(
            X=X_batch,
            obs=adata.obs.iloc[batch_idx].copy(),
            var=adata.var.copy(),
        )

        chunk_path = group_dir / f"chunk_{start // batch_size:05d}.h5ad"
        batch.write_h5ad(chunk_path, compression="gzip")
        chunk_files.append(str(chunk_path))

        del X_batch, batch

        if (start // batch_size + 1) % 100 == 0:
            done = min(start + batch_size, len(indices))
            print(f"{group_name}: {done:,}/{len(indices):,} cells saved")
            gc.collect()

    # 将分块文件在磁盘上合并，不把所有分块表达矩阵读入 RAM
    output_path = output_dir / f"KOLF_{group_name}.h5ad"
    if chunk_files:
        concat_on_disk(
            chunk_files,
            str(output_path),
            axis="0",
            join="inner",
            index_unique=None,
        )
    print(f"已生成：{output_path}")

# 关闭 backed 文件句柄
adata.file.close()
print("数据拆分完成。临时分块保存在：", temp_dir)


## 4. 输出文件说明

- `KOLF_control.h5ad`：`gene_target == "NTC"` 的细胞。
- `KOLF_perturbation.h5ad`：`gene_target != "NTC"` 的细胞。
- `temp_chunks/`：分批写入的中间文件。确认合并文件可正常打开、后续流程不再需要分块后，再手动删除临时目录以释放磁盘空间。

此流程只做拆分，不会自动做归一化、筛选高变基因或差异表达分析。